# Log4j CFG Extraction with Soot (Notebook Wrapper)

This notebook runs and inspects the CFG extraction pipeline implemented in `scripts/extract_log4j_cfg.py`. The script remains the single source of truth, so notebook experiments and command-line runs produce the same artifacts.

The pipeline compiles legacy Log4j 1.0 Java source, extracts statement-level method CFGs with Soot `ExceptionalUnitGraph`, and aggregates methods into file-level graphs aligned with the PROMISE dataset labels.

## 0) Setup Paths and Imports

In [11]:
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd

# Resolve repo root whether the notebook is run from repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCRIPT_PATH = REPO_ROOT / 'scripts' / 'extract_log4j_cfg.py'
CFG_OUTPUT_DIR = REPO_ROOT / 'outputs' / 'log4j' / 'cfg'

print('Repository root:', REPO_ROOT)
print('Extraction script:', SCRIPT_PATH)
print('CFG output directory:', CFG_OUTPUT_DIR)

Repository root: /Users/iman/Desktop/python_sdp_gnn
Extraction script: /Users/iman/Desktop/python_sdp_gnn/scripts/extract_log4j_cfg.py
CFG output directory: /Users/iman/Desktop/python_sdp_gnn/outputs/log4j/cfg


## 1) Run CFG Extraction

The extractor reads `outputs/log4j/log4j_name_to_source_mapping.csv`, compiles the legacy Java source with ECJ, and runs the Soot backend. Soot converts each concrete method body to Jimple and builds an `ExceptionalUnitGraph`.

It exports:
- readable file-level graph JSON files
- structural node feature tensors
- exact CFG node type id tensors
- edge index and edge type tensors
- stable node and edge type vocabularies
- a summary and markdown report

In [12]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print('Compiler/Soot diagnostics were captured. Character count:', len(result.stderr))

CFG extraction finished (backend=soot). graphs_generated=119 parse_failures=2

Compiler/Soot diagnostics were captured. Character count: 1040837


## 2) Inspect Extraction Summary

In [13]:
summary = json.loads((CFG_OUTPUT_DIR / 'cfg_summary.json').read_text())
summary

{'requested_mapped_files': 119,
 'graphs_generated': 119,
 'soot_graphs': 117,
 'placeholder_graphs': 2,
 'parse_failures': 2,
 'validation_issues': 0,
 'total_nodes': 4945,
 'total_edges': 4232,
 'total_methods': 729,
 'avg_nodes_per_file': 41.554621848739494,
 'avg_edges_per_file': 35.563025210084035,
 'node_feature_dim': 3,
 'node_type_vocab_size': 10,
 'edge_type_vocab_size': 5,
 'backend': 'soot'}

In [14]:
cfg_index = pd.read_csv(CFG_OUTPUT_DIR / 'cfg_index.csv')
print('Generated graphs:', len(cfg_index))
print()
print('Extraction modes:')
print(cfg_index['extraction_mode'].value_counts())
cfg_index.head()

Generated graphs: 119

Extraction modes:
extraction_mode
soot           117
placeholder      2
Name: count, dtype: int64


,file_id,source_path,extraction_mode,num_nodes,num_edges,num_methods,node_feature_dim,graph_json,x_npy,node_type_id_npy,edge_index_npy,edge_type_npy
0,org.apache.log4j.Appender,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,soot,5,4,1,3,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
1,org.apache.log4j.AppenderSkeleton,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,soot,83,70,13,3,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
2,org.apache.log4j.AsyncAppender,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,soot,79,67,12,3,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
3,org.apache.log4j.BasicConfigurator,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,soot,67,55,12,3,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...
4,org.apache.log4j.Category,/Users/iman/Desktop/python_sdp_gnn/projects/lo...,soot,328,281,47,3,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...,/Users/iman/Desktop/python_sdp_gnn/outputs/log...


In [15]:
parse_failures = json.loads((CFG_OUTPUT_DIR / 'parse_failures.json').read_text())
pd.DataFrame(parse_failures)

,file_id,source_path,error
0,__global__,,javac_partial_failure:255
1,org.apache.log4j.AppenderSkeleton,,java.lang.NullPointerException: Cannot invoke ...


## 3) Understand the CFG Tensors

Each CFG node is stored in two parts, and each edge is stored in two aligned tensors:

```text
node_type_id.npy -> exact CFG node type id
x.npy            -> [source_line, has_snippet, is_synthetic]
edge_index.npy   -> [source_node_ids, target_node_ids]
edge_type.npy    -> exact edge type id for each edge_index column
```

The learnable embeddings are not stored in the dataset. During GNN training, concatenate a learned node type embedding with the three extracted node features. Use edge type ids with relation-aware message passing or edge embeddings.

In [16]:
node_type_vocab = json.loads((CFG_OUTPUT_DIR / 'node_type_vocab.json').read_text())
edge_type_vocab = json.loads((CFG_OUTPUT_DIR / 'edge_type_vocab.json').read_text())
id_to_node_type = {node_type_id: node_type for node_type, node_type_id in node_type_vocab.items()}
id_to_edge_type = {edge_type_id: edge_type for edge_type, edge_type_id in edge_type_vocab.items()}

print('CFG node type vocabulary:')
display(pd.DataFrame(sorted(node_type_vocab.items(), key=lambda item: item[1]), columns=['node_type', 'node_type_id']))
print('CFG edge type vocabulary:')
display(pd.DataFrame(sorted(edge_type_vocab.items(), key=lambda item: item[1]), columns=['edge_type', 'edge_type_id']))

CFG node type vocabulary:


,node_type,node_type_id
0,ENTRY,0
1,EXIT,1
2,STATEMENT,2
3,CONDITION,3
4,RETURN,4
5,THROW,5
6,LOOP,6
7,SWITCH,7
8,CATCH,8
9,FINALLY,9


CFG edge type vocabulary:


,edge_type,edge_type_id
0,CFG_NEXT,0
1,CFG_TRUE,1
2,CFG_FALSE,2
3,CFG_RETURN,3
4,CFG_EXCEPTION,4


## 4) Inspect One Example Graph

In [17]:
EXAMPLE_CLASS = 'org.apache.log4j.helpers.BoundedFIFO'
tensor_dir = CFG_OUTPUT_DIR / 'tensors'

graph = json.loads((CFG_OUTPUT_DIR / 'graphs' / f'{EXAMPLE_CLASS}.json').read_text())
structural_x = np.load(tensor_dir / f'{EXAMPLE_CLASS}_x.npy')
node_type_ids = np.load(tensor_dir / f'{EXAMPLE_CLASS}_node_type_id.npy')
edge_index = np.load(tensor_dir / f'{EXAMPLE_CLASS}_edge_index.npy')
edge_type_ids = np.load(tensor_dir / f'{EXAMPLE_CLASS}_edge_type.npy')

print('Class:', EXAMPLE_CLASS)
print('methods:', len(graph['methods']))
print('structural_x shape:', structural_x.shape)
print('node_type_ids shape:', node_type_ids.shape)
print('edge_index shape:', edge_index.shape)
print('edge_type_ids shape:', edge_type_ids.shape)

Class: org.apache.log4j.helpers.BoundedFIFO
methods: 8
structural_x shape: (80, 3)
node_type_ids shape: (80,)
edge_index shape: (2, 79)
edge_type_ids shape: (79,)


In [18]:
node_preview = pd.DataFrame({
    'node_id': np.arange(len(node_type_ids)),
    'node_type_id': node_type_ids,
    'node_type': [id_to_node_type[int(node_type_id)] for node_type_id in node_type_ids],
    'source_line': structural_x[:, 0],
    'has_snippet': structural_x[:, 1],
    'is_synthetic': structural_x[:, 2],
})
node_preview.head(20)

,node_id,node_type_id,node_type,source_line,has_snippet,is_synthetic
0,0,0,ENTRY,0.0,1.0,1.0
1,1,1,EXIT,0.0,1.0,1.0
2,2,2,STATEMENT,0.0,1.0,0.0
3,3,2,STATEMENT,0.0,1.0,0.0
4,4,2,STATEMENT,31.0,1.0,0.0
5,5,2,STATEMENT,31.0,1.0,0.0
6,6,5,THROW,31.0,1.0,0.0
7,7,0,ENTRY,0.0,1.0,1.0
8,8,1,EXIT,0.0,1.0,1.0
9,9,2,STATEMENT,0.0,1.0,0.0


In [19]:
edge_preview = pd.DataFrame({
    'source_node_id': edge_index[0],
    'target_node_id': edge_index[1],
    'edge_type_id': edge_type_ids,
    'edge_type': [id_to_edge_type[int(edge_type_id)] for edge_type_id in edge_type_ids],
})
edge_preview.head(25)

,source_node_id,target_node_id,edge_type_id,edge_type
0,0,2,0,CFG_NEXT
1,2,3,0,CFG_NEXT
2,3,4,0,CFG_NEXT
3,4,5,0,CFG_NEXT
4,5,6,0,CFG_NEXT
5,6,1,4,CFG_EXCEPTION
6,7,9,0,CFG_NEXT
7,9,10,0,CFG_NEXT
8,10,11,0,CFG_NEXT
9,11,12,2,CFG_FALSE


## 5) Assemble Features Inside the GNN

The extraction output is designed for a relation-aware GNN. The model learns both node type embeddings and, if the selected GNN layer supports them, edge type embeddings.

```python
import torch
from torch import nn

node_type_embedding = nn.Embedding(num_embeddings=10, embedding_dim=32)
edge_type_embedding = nn.Embedding(num_embeddings=5, embedding_dim=8)

complete_node_x = torch.cat([node_type_embedding(node_type_id), structural_x], dim=1)
edge_x = edge_type_embedding(edge_type_id)
# complete_node_x shape: [num_nodes, 35]
```

After message passing, pool node embeddings into one CFG representation per Java class. That file-level representation can later be fused with AST and NDG representations before defect classification.

## 6) Inspect the Generated Report

In [20]:
report_text = (CFG_OUTPUT_DIR / 'cfg_report.md').read_text()

try:
    from IPython.display import Markdown, display
    display(Markdown(report_text))
except ImportError:
    print(report_text)

# CFG Extraction Report for Log4j 1.0

## 1. Objective
This report describes the CFG graph extraction step for the Log4j 1.0 PROMISE dataset used in the multi-view software defect prediction pipeline. The goal is to create method-level control-flow graphs from compiled Java source and aggregate them into one graph per mapped Java class so they remain aligned with the file-level PROMISE labels.

## 2. Input Data
- Dataset: `projects/log4j/log4j-1.0.csv`
- Preprocessed dataset: `outputs/log4j/log4j_preprocessed_standard.csv`
- Class-to-source mapping: `outputs/log4j/log4j_name_to_source_mapping.csv`
- Java source root: `projects/log4j/logging-log4j1-v_1_0/src/java`
- Mapped source files requested for CFG extraction: 119

## 3. Extraction Pipeline
- The Java sources are compiled with Eclipse ECJ in Java 1.3 compatibility mode because Log4j 1.0 contains identifiers that became reserved words in newer Java versions.
- Soot loads the generated class files and retrieves each concrete method or constructor body.
- Soot `ExceptionalUnitGraph` builds the control-flow graph for each concrete method.
- Each method graph receives explicit synthetic `ENTRY` and `EXIT` nodes.
- Soot Jimple units become statement-level CFG nodes.
- Normal, conditional, return, and exceptional flow relations become typed CFG edges.
- All method graphs from the same Java class are aggregated into one file-level graph.
- The AST extractor is independent and remains unchanged.

```text
Java source -> ECJ compile -> Soot Jimple -> ExceptionalUnitGraph -> method CFGs -> file-level CFG
```

## 4. CFG Representation
The CFG view stores control behavior mainly in typed edges. Node features remain compact so future AST, CFG, NDG, control-dependency, and call relations can be integrated without duplicating behavior inside node feature vectors.

### 4.1 Node Types
| Node type | ID |
| --- | ---: |
| `ENTRY` | 0 |
| `EXIT` | 1 |
| `STATEMENT` | 2 |
| `CONDITION` | 3 |
| `RETURN` | 4 |
| `THROW` | 5 |
| `LOOP` | 6 |
| `SWITCH` | 7 |
| `CATCH` | 8 |
| `FINALLY` | 9 |

`LOOP`, `CATCH`, and `FINALLY` remain reserved vocabulary entries for compatibility. The current Soot Jimple extractor represents their behavior through statement nodes and typed graph edges.

### 4.2 Edge Types
| Edge type | ID |
| --- | ---: |
| `CFG_NEXT` | 0 |
| `CFG_TRUE` | 1 |
| `CFG_FALSE` | 2 |
| `CFG_RETURN` | 3 |
| `CFG_EXCEPTION` | 4 |

| Edge type | Meaning |
| --- | --- |
| `CFG_NEXT` | Normal control-flow successor. |
| `CFG_TRUE` | Target reached when an `if` condition evaluates true. |
| `CFG_FALSE` | Target reached when an `if` condition evaluates false. |
| `CFG_RETURN` | Return statement flow to the method `EXIT`. |
| `CFG_EXCEPTION` | Exceptional successor or explicit throw flow. |

### 4.3 Stored Node Metadata
- `file_id`, `method_id`, `id`
- `node_type`, `node_type_id`
- `line_start`, `line_end` when Soot exposes source line metadata
- `snippet`: normalized Jimple statement text
- `is_synthetic`: true for generated `ENTRY`, `EXIT`, and placeholder nodes

## 5. What the Tensor Files Mean
A tensor is a numeric array used by machine learning frameworks. Each file-level CFG is exported as:

| File | Shape | Meaning |
| --- | --- | --- |
| `*_x.npy` | `[num_nodes, 3]` | Structural features: source line number, has-snippet flag, and is-synthetic flag. |
| `*_node_type_id.npy` | `[num_nodes]` | Exact CFG node type ids consumed by a trainable embedding layer. |
| `*_edge_index.npy` | `[2, num_edges]` | Connectivity. Row 0 stores source ids; row 1 stores target ids. |
| `*_edge_type.npy` | `[num_edges]` | Exact edge type id for each matching column in `edge_index`. |

During model training, convert each `node_type_id` to a learned embedding and concatenate it with the three stored structural values. Use `edge_type` for relation-aware message passing.

```text
complete_node_x = concat(node_type_embedding(node_type_id), structural_x)
edge_index[:, i] and edge_type[i] describe the same directed CFG edge
```

## 6. How to Use the CFG Tensors in a GNN
At model level, the processing steps are:

1. Load `node_type_id`, structural `x`, `edge_index`, and `edge_type` for each CFG graph.
2. Convert every CFG node type id to a learned embedding.
3. Concatenate each embedding with its three fixed structural values.
4. Use `edge_type` with relation-aware GNN layers or edge embeddings.
5. Pool node representations into one graph representation for the Java class.
6. Join the pooled CFG representation with the AST and future NDG view representations.
7. Feed the fused representation into a defect classifier.

Minimal PyTorch-style preparation:

```python
import torch
from torch import nn

node_type_embedding = nn.Embedding(num_embeddings=10, embedding_dim=32)
edge_type_embedding = nn.Embedding(num_embeddings=5, embedding_dim=8)

complete_node_x = torch.cat([node_type_embedding(node_type_id), structural_x], dim=1)
edge_x = edge_type_embedding(edge_type_id)
```

## 7. Output Files
- `outputs/log4j/cfg/cfg_index.csv`: index of generated file-level graphs and tensor paths.
- `outputs/log4j/cfg/graphs/*.json`: readable file-level graphs with method, node, and edge metadata.
- `outputs/log4j/cfg/tensors/*_x.npy`: structural node features.
- `outputs/log4j/cfg/tensors/*_node_type_id.npy`: exact CFG node type ids.
- `outputs/log4j/cfg/tensors/*_edge_index.npy`: directed edge connectivity.
- `outputs/log4j/cfg/tensors/*_edge_type.npy`: edge relation type ids.
- `outputs/log4j/cfg/node_type_vocab.json`: stable CFG node type vocabulary.
- `outputs/log4j/cfg/edge_type_vocab.json`: stable CFG edge type vocabulary.
- `outputs/log4j/cfg/cfg_summary.json`: global extraction statistics.
- `outputs/log4j/cfg/parse_failures.json`: compile and Soot extraction issues.
- `outputs/log4j/cfg/validation_issues.json`: CFG validation issue log.

## 8. Results

| Metric | Value |
| --- | ---: |
| Mapped samples requested | 119 |
| File-level graphs generated | 119 |
| Real Soot graphs | 117 |
| Placeholder graphs | 2 |
| Logged compile/extraction issues | 2 |
| Total method CFGs | 729 |
| Total nodes | 4945 |
| Total edges | 4232 |
| Average nodes per graph | 41.55 |
| Average edges per graph | 35.56 |
| Extracted structural feature dimension | 3 |
| CFG node type vocabulary size | 10 |
| CFG edge type vocabulary size | 5 |

## 9. Node and Edge Distribution
### 9.1 Node Types
- `STATEMENT`: 2739
- `ENTRY`: 729
- `EXIT`: 729
- `THROW`: 707
- `RETURN`: 25
- `CONDITION`: 16

### 9.2 Edge Types
- `CFG_NEXT`: 3468
- `CFG_EXCEPTION`: 707
- `CFG_RETURN`: 25
- `CFG_FALSE`: 16
- `CFG_TRUE`: 16

## 10. Notes and Limitations
- The extractor uses Soot CFGs from compiled class files. It does not construct CFGs from `javalang` AST nodes.
- Soot operates on Jimple statements, so snippets are normalized intermediate-representation statements, not exact source substrings.
- Two mapped interface-like classes currently receive placeholder `ENTRY -> EXIT` graphs because no concrete method body is available. Keep this distinction visible during experiments.
- Log4j 1.0 is legacy Java source. ECJ emits compile diagnostics under the modern Java runtime while still producing analyzable class files with `-proceedOnError`.
- One Soot method body retrieval issue is logged for `AppenderSkeleton`; other recoverable methods in that class remain included.
- `CFG_BREAK` and `CFG_CONTINUE` are intentionally not exported as separate edge types by this backend. Soot resolves them to normal control-flow successors after compilation.

### 10.1 Logged Issues
| File | Issue |
| --- | --- |
| `__global__` | `javac_partial_failure:255` |
| `org.apache.log4j.AppenderSkeleton` | `java.lang.NullPointerException: Cannot invoke "soot.Body.getUnits()" because "body" is null` |

## 11. Sample Visualization
The following diagram shows the Soot CFG for `BoundedFIFO.get#1[p0]`. It is limited to a compact method-level example so the control-flow relations remain readable.

```mermaid
graph TD
  N7["ENTRY#7"]
  N8["EXIT#8"]
  N9["STATEMENT#9 | r0 := @this: org.apache.log4j.helpers.BoundedFIFO"]
  N10["STATEMENT#10 | $i0 = r0.<org.apache.log4j.helpers.BoundedFIFO: int num"]
  N11["CONDITION#11 | if $i0 != 0 goto $r1 = r0.<org.apache.log4j.helpers.Bou"]
  N12["RETURN#12 | return null"]
  N13["STATEMENT#13 | $r1 = r0.<org.apache.log4j.helpers.BoundedFIFO: org.apa"]
  N14["STATEMENT#14 | $i1 = r0.<org.apache.log4j.helpers.BoundedFIFO: int fir"]
  N15["STATEMENT#15 | r2 = $r1[$i1]"]
  N16["STATEMENT#16 | $i2 = r0.<org.apache.log4j.helpers.BoundedFIFO: int fir"]
  N17["STATEMENT#17 | $i3 = $i2 + 1"]
  N18["STATEMENT#18 | r0.<org.apache.log4j.helpers.BoundedFIFO: int first> = "]
  N19["STATEMENT#19 | $i4 = r0.<org.apache.log4j.helpers.BoundedFIFO: int max"]
  N20["CONDITION#20 | if $i3 != $i4 goto $i5 = r0.<org.apache.log4j.helpers.B"]
  N21["STATEMENT#21 | r0.<org.apache.log4j.helpers.BoundedFIFO: int first> = "]
  N22["STATEMENT#22 | $i5 = r0.<org.apache.log4j.helpers.BoundedFIFO: int num"]
  N23["STATEMENT#23 | $i6 = $i5 - 1"]
  N24["STATEMENT#24 | r0.<org.apache.log4j.helpers.BoundedFIFO: int numElemen"]
  N25["RETURN#25 | return r2"]
  N7 -->|CFG_NEXT| N9
  N9 -->|CFG_NEXT| N10
  N10 -->|CFG_NEXT| N11
  N11 -->|CFG_FALSE| N12
  N11 -->|CFG_TRUE| N13
  N12 -->|CFG_RETURN| N8
  N13 -->|CFG_NEXT| N14
  N14 -->|CFG_NEXT| N15
  N15 -->|CFG_NEXT| N16
  N16 -->|CFG_NEXT| N17
  N17 -->|CFG_NEXT| N18
  N18 -->|CFG_NEXT| N19
  N19 -->|CFG_NEXT| N20
  N20 -->|CFG_FALSE| N21
  N20 -->|CFG_TRUE| N22
  N21 -->|CFG_NEXT| N22
  N22 -->|CFG_NEXT| N23
  N23 -->|CFG_NEXT| N24
  N24 -->|CFG_NEXT| N25
  N25 -->|CFG_RETURN| N8
```

## 12. Future Integration
The file-level graph boundary matches the AST pipeline and the PROMISE labels. Future multi-view graph integration can unify relations such as:

```text
AST_CHILD, CFG_NEXT, CFG_TRUE, CFG_FALSE, CFG_RETURN, CFG_EXCEPTION, DATA_DEP, CONTROL_DEP, CALL
```
